# Amazon Product Review Sentiment Analysis (Kaggle)
## Beginner-friendly BERT fine-tuning vs LSTM

**Dataset:** [Amazon Fine Food Reviews – Kaggle](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

This notebook downloads `Reviews.csv` from Kaggle and uses:
- `Text` as the review
- `Score` as the star rating
- 1–2 stars = Negative, 3 stars = Neutral, 4–5 stars = Positive

It trains an LSTM baseline and fine-tunes BERT on the same training data, then compares test accuracy and macro-F1.

**Recommended:** Run in Google Colab with a GPU. The notebook uses a small balanced sample to keep training manageable. Increase the sample sizes if you have more GPU time.

## 1. Install required libraries

In [ ]:
!pip -q install kagglehub transformers scikit-learn pandas matplotlib

## 2. Import the Kaggle dataset directly with Python

We use **KaggleHub** to download the public Kaggle dataset directly into the notebook runtime. No local CSV upload and no manual `kaggle.json` upload step are needed for this public dataset.

Dataset: [Amazon Fine Food Reviews – Kaggle](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

KaggleHub uses `dataset_download()` to retrieve the dataset. Public datasets generally download without authentication unless Kaggle requires user consent.

In [ ]:
import kagglehub
import os
import glob

# Download the public Kaggle dataset directly.
dataset_path = kagglehub.dataset_download(
    "snap/amazon-fine-food-reviews"
)

print("Dataset downloaded to:", dataset_path)

## 3. Load and prepare the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Find Reviews.csv anywhere inside the downloaded Kaggle dataset directory.
csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
reviews_files = [
    path for path in csv_files
    if os.path.basename(path).lower() == "reviews.csv"
]

if reviews_files:
    csv_path = reviews_files[0]
elif csv_files:
    # Fallback if the dataset file has a different CSV name.
    csv_path = csv_files[0]
else:
    raise FileNotFoundError(
        f"No CSV file found inside the downloaded dataset: {dataset_path}"
    )

print("Using file:", csv_path)

df = pd.read_csv(csv_path, encoding="ISO-8859-1")
df = df[["Text", "Score"]].dropna()
df["Text"] = df["Text"].astype(str).str.strip()
df = df[df["Text"] != ""]

def make_label(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df["label"] = df["Score"].apply(make_label)

print(df.head())
print("\nClass counts:")
print(df["label"].value_counts().sort_index())
print("\n0 = Negative, 1 = Neutral, 2 = Positive")

## 4. Create a small balanced sample and split it

In [ ]:
from sklearn.model_selection import train_test_split

# Take up to 2,000 examples from each sentiment class.
# Increase this number if you have enough GPU memory and time.
SAMPLES_PER_CLASS = 2000

parts = []
for label in [0, 1, 2]:
    class_df = df[df["label"] == label]
    parts.append(class_df.sample(
        n=min(SAMPLES_PER_CLASS, len(class_df)),
        random_state=42
    ))

data = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    data["Text"].tolist(),
    data["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

print("Train reviews:", len(X_train))
print("Test reviews:", len(X_test))

## 5. Train the LSTM baseline

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.random.set_seed(42)

word_tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
word_tokenizer.fit_on_texts(X_train)

X_train_seq = word_tokenizer.texts_to_sequences(X_train)
X_test_seq = word_tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=100, padding="post", truncating="post")

lstm_model = Sequential([
    Embedding(input_dim=10000, output_dim=64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.fit(
    X_train_pad, np.array(y_train),
    epochs=3,
    batch_size=32,
    validation_split=0.1
)

## 6. Evaluate LSTM

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

lstm_prob = lstm_model.predict(X_test_pad)
lstm_pred = np.argmax(lstm_prob, axis=1)

print("LSTM accuracy:", accuracy_score(y_test, lstm_pred))
print("LSTM macro F1:", f1_score(y_test, lstm_pred, average="macro"))
print(classification_report(
    y_test, lstm_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

## 7. Fine-tune BERT

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

MODEL_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = bert_tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=96,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "input_ids": self.encodings["input_ids"][index],
            "attention_mask": self.encodings["attention_mask"][index],
            "labels": self.labels[index]
        }

train_dataset = ReviewDataset(X_train, y_train)
test_dataset = ReviewDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(device)

optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

In [ ]:
# One epoch keeps this beginner notebook reasonably quick.
# Increase to 2–3 epochs for a stronger experiment.
EPOCHS = 1

for epoch in range(EPOCHS):
    bert_model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader):.4f}")

## 8. Evaluate BERT

In [ ]:
bert_model.eval()
bert_predictions = []
actual_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        predictions = torch.argmax(outputs.logits, dim=1)

        bert_predictions.extend(predictions.cpu().numpy())
        actual_labels.extend(batch["labels"].numpy())

print("BERT accuracy:", accuracy_score(actual_labels, bert_predictions))
print("BERT macro F1:", f1_score(actual_labels, bert_predictions, average="macro"))
print(classification_report(
    actual_labels, bert_predictions,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

## 9. Compare LSTM and BERT

In [ ]:
results = pd.DataFrame({
    "Model": ["LSTM", "BERT"],
    "Accuracy": [
        accuracy_score(y_test, lstm_pred),
        accuracy_score(actual_labels, bert_predictions)
    ],
    "Macro F1": [
        f1_score(y_test, lstm_pred, average="macro"),
        f1_score(actual_labels, bert_predictions, average="macro")
    ]
})

display(results)

results.set_index("Model").plot(kind="bar", ylim=(0, 1), figsize=(7, 4))
plt.title("LSTM vs BERT")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

## 10. Test your own review

In [ ]:
def predict_review(review):
    # BERT prediction
    bert_model.eval()
    inputs = bert_tokenizer(
        review, return_tensors="pt",
        truncation=True, padding=True, max_length=96
    ).to(device)

    with torch.no_grad():
        bert_output = bert_model(**inputs).logits
    bert_label = int(torch.argmax(bert_output, dim=1).item())

    # LSTM prediction
    sequence = word_tokenizer.texts_to_sequences([review])
    padded = pad_sequences(sequence, maxlen=100, padding="post", truncating="post")
    lstm_label = int(np.argmax(lstm_model.predict(padded, verbose=0), axis=1)[0])

    labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
    print("Review:", review)
    print("BERT:", labels[bert_label])
    print("LSTM:", labels[lstm_label])

predict_review("This product is excellent and works perfectly!")

## 11. Conclusion

- **LSTM** reads a review as a sequence and learns sentiment patterns from the Kaggle training data.
- **BERT** starts with language knowledge learned during pre-training and is fine-tuned on the same three sentiment classes.
- Transfer learning saves the effort of training a large language model from scratch.
- BERT may provide stronger contextual understanding, but can need more memory, compute and inference time.
- Real deployment must consider latency, model size, privacy, class imbalance, long reviews and changing customer language.

**For your report:** Record the accuracy, macro-F1, training time, GPU type and sample size from your actual run. Do not copy assumed results. The rating-to-sentiment mapping is a proxy: a three-star review is not always neutral, and four/five-star reviews can still contain criticism.

**Dataset citation:** Kaggle, *Amazon Fine Food Reviews* – https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews